<a href="https://colab.research.google.com/github/mustafa-ahsan/ultrasound-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from google.colab import files
from PIL import Image
import io

print("🩺 أداة السونار عالي الدقة (تحديث: بدون تقطيع)")
print("الرجاء رفع صورة سونار من جهازك...")

uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    image_bytes = uploaded[file_name]

    try:
        # قراءة الصورة وتوحيد صيغتها لتجنب الأخطاء
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        img_array = np.array(img)
        img_gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

        print("✅ تم رفع الصورة بنجاح! عدل براحتك الآن (بدون بطء):")

        def sharpen_ultrasound(details_level, sharpness, theme):
            # 1. إبراز التفاصيل (CLAHE)
            clahe = cv2.createCLAHE(clipLimit=details_level, tileGridSize=(8,8))
            detailed_img = clahe.apply(img_gray)

            # 2. زيادة الحدة (Unsharp Masking)
            if sharpness > 0:
                blurred = cv2.GaussianBlur(detailed_img, (0, 0), 3)
                sharpened = cv2.addWeighted(detailed_img, 1.0 + sharpness, blurred, -sharpness, 0)
            else:
                sharpened = detailed_img

            # 3. التلوين الحراري
            if theme == 'Heatmap (Jet)':
                cmap_type = cv2.COLORMAP_JET
            elif theme == 'Bone (X-Ray style)':
                cmap_type = cv2.COLORMAP_BONE
            else:
                cmap_type = cv2.COLORMAP_MAGMA

            colored = cv2.applyColorMap(sharpened, cmap_type)
            colored_rgb = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)

            # 4. عرض النتائج بدقة عالية
            plt.figure(figsize=(18, 6))

            plt.subplot(1, 3, 1)
            plt.imshow(img_gray, cmap='gray')
            plt.title('1. Original ')
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(sharpened, cmap='gray')
            plt.title(f'2. Enhanced Details\n(CLAHE: {details_level}, Sharpness: {sharpness})')
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(colored_rgb)
            plt.title(f'3. Colorized\n(Theme: {theme})')
            plt.axis('off')

            plt.tight_layout()
            plt.show()

        # تشغيل الواجهة التفاعلية (مع إضافة continuous_update=False لمنع التعليق)
        interact(sharpen_ultrasound,
                 details_level=widgets.FloatSlider(min=1.0, max=5.0, step=0.5, value=2.0, description='عمق التفاصيل:', continuous_update=False),
                 sharpness=widgets.FloatSlider(min=0.0, max=3.0, step=0.5, value=1.0, description='حدة الحواف:', continuous_update=False),
                 theme=widgets.Dropdown(options=['Heatmap (Jet)', 'Bone (X-Ray style)', 'Magma'], value='Heatmap (Jet)', description='الثيم:'))

    except Exception as e:
        print("❌ حدث خطأ في معالجة الصورة. تأكد من رفع ملف صورة صالح.")
        print("تفاصيل الخطأ:", e)
else:
    print("⚠️ لم يتم رفع أي صورة. يرجى إعادة تشغيل الخلية.")

🩺 أداة السونار عالي الدقة (تحديث: بدون تقطيع)
الرجاء رفع صورة سونار من جهازك...


KeyboardInterrupt: 